## **Visualisasi Keypoint SIFT**

In [ ]:
import cv2
import matplotlib.pyplot as plt

img = cv2.imread('/content/siman_1.png')

img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

sift = cv2.SIFT_create()

keypoints, descriptors = sift.detectAndCompute(gray, None)

print("Jumlah keypoints:", len(keypoints))

img_kp = cv2.drawKeypoints(
    img_rgb,
    keypoints,
    None,
    flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
)

plt.figure(figsize=(10, 6))
plt.imshow(img_kp)
plt.title("SIFT Keypoints (Scale & Orientation)")
plt.axis('off')
plt.show()


## **DoG**

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

img = cv2.imreadimg = cv2.imread('/content/siman_1.png')
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

sigma = 1.6
k = 1.6
sigma_k = sigma * k

blur_sigma = cv2.GaussianBlur(gray, (0, 0), sigmaX=sigma)

blur_sigma_k = cv2.GaussianBlur(gray, (0, 0), sigmaX=sigma_k)

dog = cv2.subtract(blur_sigma_k, blur_sigma)

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(blur_sigma, cmap='gray')
plt.title(r"Gaussian Blur ($\sigma$)")
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(blur_sigma_k, cmap='gray')
plt.title(r"Gaussian Blur ($k\sigma$)")
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(dog, cmap='gray')
plt.title("Difference of Gaussian (DoG)")
plt.axis('off')

plt.tight_layout()
plt.show()


## **Feature Matching**

In [ ]:
# # SIFT (Scale-Invariant Feature Transform)

# ## Import resources and display image

import cv2
import matplotlib.pyplot as plt
import numpy as np

# Load the image
img1 = cv2.imread('/content/siman_1.png')
img2 = cv2.imread('/content/siman_2.png')

# Convert BGR to RGB (for matplotlib)
img1_rgb = cv2.cvtColor(img1, cv2.COLOR_BGR2RGB)
img2_rgb = cv2.cvtColor(img2, cv2.COLOR_BGR2RGB)

# Convert to grayscale
gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)

sift = cv2.SIFT_create()

kp1, des1 = sift.detectAndCompute(gray1, None)
kp2, des2 = sift.detectAndCompute(gray2, None)

print("Keypoints img1:", len(kp1))
print("Keypoints img2:", len(kp2))

bf = cv2.BFMatcher(cv2.NORM_L2)

matches = bf.knnMatch(des1, des2, k=2)

good_matches = []
ratio = 0.75
for m, n in matches:
    if m.distance < ratio * n.distance:
        good_matches.append(m)

print("Good matches:", len(good_matches))

matched_img = cv2.drawMatches(
    img1_rgb, kp1,
    img2_rgb, kp2,
    good_matches[:100], None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
)

plt.figure(figsize=(14, 7))
plt.imshow(matched_img)
plt.axis('off')
plt.title("SIFT Feature Matching")
plt.show()

## **Panorama Sederhana**

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

img1 = cv2.imread('/content/siman_1.png')
img2 = cv2.imread('/content/siman_2.png')

img1_rgb = cv2.cvtColor(img1, cv2.COLOR_BGR2RGB)
img2_rgb = cv2.cvtColor(img2, cv2.COLOR_BGR2RGB)

gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)

sift = cv2.SIFT_create()

kp1, des1 = sift.detectAndCompute(gray1, None)
kp2, des2 = sift.detectAndCompute(gray2, None)

print("Keypoints img1:", len(kp1))
print("Keypoints img2:", len(kp2))

bf = cv2.BFMatcher(cv2.NORM_L2)
matches = bf.knnMatch(des1, des2, k=2)

good_matches = []
for m, n in matches:
    if m.distance < 0.75 * n.distance:
        good_matches.append(m)

print("Good matches:", len(good_matches))

pts1 = np.float32([kp1[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
pts2 = np.float32([kp2[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)

H, _ = cv2.findHomography(pts1, pts2, cv2.RANSAC, 5.0)

h1, w1 = img1.shape[:2]
h2, w2 = img2.shape[:2]

corners_img1 = np.float32([
    [0, 0],
    [w1, 0],
    [w1, h1],
    [0, h1]
]).reshape(-1, 1, 2)

corners_img2 = np.float32([
    [0, 0],
    [w2, 0],
    [w2, h2],
    [0, h2]
]).reshape(-1, 1, 2)

warped_corners_img1 = cv2.perspectiveTransform(corners_img1, H)

all_corners = np.concatenate((warped_corners_img1, corners_img2), axis=0)

x_min, y_min = np.int32(all_corners.min(axis=0).ravel())
x_max, y_max = np.int32(all_corners.max(axis=0).ravel())

panorama_width  = x_max - x_min
panorama_height = y_max - y_min

translation = np.array([
    [1, 0, -x_min],
    [0, 1, -y_min],
    [0, 0, 1]
])

panorama = cv2.warpPerspective(
    img1,
    translation @ H,
    (panorama_width, panorama_height)
)

panorama[-y_min:h2 - y_min, -x_min:w2 - x_min] = img2

panorama_rgb = cv2.cvtColor(panorama, cv2.COLOR_BGR2RGB)

matches_vis = cv2.drawMatches(
    img1_rgb, kp1,
    img2_rgb, kp2,
    good_matches[:100],   # batasi agar tidak terlalu padat
    None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
)

plt.figure(figsize=(18, 12))

plt.subplot(3, 2, 1)
plt.imshow(img1_rgb)
plt.title("Before: Image 1 (siman_1.jpg)")
plt.axis('off')

plt.subplot(3, 2, 2)
plt.imshow(img2_rgb)
plt.title("Before: Image 2 (siman_2.jpg)")
plt.axis('off')


plt.subplot(3, 2, (3, 4))
plt.imshow(matches_vis)
plt.title("SIFT Feature Matches (Stitches)")
plt.axis('off')

plt.subplot(3, 2, (5, 6))
plt.imshow(panorama_rgb)
plt.title("After: Panorama (Expanded Canvas)")
plt.axis('off')

plt.tight_layout()
plt.show()


# **Forgery Detection (deteksi kemiripan citra)**

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Load image (grayscale)
img1 = cv2.imread("/content/siman_1.png", cv2.IMREAD_GRAYSCALE)
img2 = cv2.imread("/content/ujicoba.jpg", cv2.IMREAD_GRAYSCALE)

# Inisialisasi SIFT
sift = cv2.SIFT_create()

# Deteksi keypoints & descriptors
kp1, des1 = sift.detectAndCompute(img1, None)
kp2, des2 = sift.detectAndCompute(img2, None)

# Feature matching (FLANN)
FLANN_INDEX_KDTREE = 1
index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
search_params = dict(checks=50)

flann = cv2.FlannBasedMatcher(index_params, search_params)
matches = flann.knnMatch(des1, des2, k=2)

# Lowe's Ratio Test
good_matches = []
for m, n in matches:
    if m.distance < 0.75 * n.distance:
        good_matches.append(m)

# Hitung tingkat kemiripan
similarity = (len(good_matches) / min(len(kp1), len(kp2))) * 100

print(f"Jumlah keypoints image 1: {len(kp1)}")
print(f"Jumlah keypoints image 2: {len(kp2)}")
print(f"Good matches: {len(good_matches)}")
print(f"Tingkat kemiripan: {similarity:.2f}%")

# Visualisasi hasil matching
result = cv2.drawMatches(
    img1, kp1, img2, kp2,
    good_matches, None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
)

plt.figure(figsize=(12,6))
plt.imshow(result, cmap='gray')
plt.title("Forgery Detection using SIFT")
plt.axis("off")
plt.show()


## ORB

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# ==============================
# LOAD IMAGE (GRAYSCALE)
# ==============================
img1 = cv2.imread("/content/siman_1.png", cv2.IMREAD_GRAYSCALE)
img2 = cv2.imread("/content/ujicoba.jpg", cv2.IMREAD_GRAYSCALE)

if img1 is None or img2 is None:
    raise IOError("Gambar tidak ditemukan!")

# ==============================
# INISIALISASI ORB
# ==============================
orb = cv2.ORB_create(
    nfeatures=2000,
    scaleFactor=1.2,
    nlevels=8
)

# ==============================
# DETEKSI KEYPOINT & DESCRIPTOR
# ==============================
kp1, des1 = orb.detectAndCompute(img1, None)
kp2, des2 = orb.detectAndCompute(img2, None)

# ==============================
# FEATURE MATCHING (BFMatcher)
# ==============================
bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
matches = bf.match(des1, des2)

# Urutkan match terbaik
matches = sorted(matches, key=lambda x: x.distance)

# ==============================
# HITUNG TINGKAT KEMIRIPAN
# ==============================
good_matches = matches[:200]  # ambil match terbaik

similarity = (len(good_matches) / min(len(kp1), len(kp2))) * 100

# ==============================
# OUTPUT HASIL
# ==============================
print("===== HASIL ORB =====")
print(f"Jumlah keypoints image 1: {len(kp1)}")
print(f"Jumlah keypoints image 2: {len(kp2)}")
print(f"Good matches: {len(good_matches)}")
print(f"Tingkat kemiripan: {similarity:.2f}%")

# ==============================
# VISUALISASI HASIL MATCHING
# ==============================
result = cv2.drawMatches(
    img1, kp1,
    img2, kp2,
    good_matches,
    None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
)

plt.figure(figsize=(12,6))
plt.imshow(result, cmap="gray")
plt.title("ORB Feature Matching")
plt.axis("off")
plt.show()
